# 01 판다스 · 4. 공정 데이터 실습

- 강의 페이지: `Web/강좌/01_판다스/판다스_수업자료.html` → **4 공정 데이터 실습** 탭
- 1~3차시에서 배운 기능을 반도체 공정 데이터에 적용합니다.
- 위에서 아래로 순서대로 실행하세요. 다른 노트북의 변수를 사용하지 않으므로 이 파일만 열어도 실행됩니다.

## 초보자 필수 보강 실습

지금까지 배운 판다스 기능을 반도체 공정 데이터에 연결해 봅니다.
각 실습은 **직접 작성 → 힌트 → 정답 확인** 순서입니다.

초보자가 꼭 기억할 원칙은 세 가지입니다.

1. 원본 데이터는 덮어쓰지 않고 `copy()`로 복사해 작업합니다.
2. 분석 전에 크기, 열 이름, 자료형, 결측값을 먼저 확인합니다.
3. 계산 결과는 몇 행이라도 직접 확인해 예상과 맞는지 검증합니다.


### 준비 · 라이브러리 불러오기

실습에 필요한 pandas와 matplotlib을 불러오고 한글 글꼴을 설정합니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'  # 제목·축 이름 한글 표시 (Mac: 'AppleGothic')
plt.rcParams['axes.unicode_minus'] = False       # 음수(-) 기호 깨짐 방지


### 실습 데이터 준비

작은 공정 이력 표를 직접 만듭니다. `None`은 측정값이 없다는 뜻이며,
마지막 행은 중복 처리 연습을 위해 의도적으로 한 번 더 넣었습니다.


In [ ]:
process_data = {
    'lot_id': ['L001', 'L002', 'L003', 'L004', 'L005', 'L005'],
    '공정': ['식각', '증착', '식각', '세정', '증착', '증착'],
    '온도_섭씨': [298.5, 302.1, None, 299.8, 305.2, 305.2],
    '압력_Pa': [101.2, 99.8, 103.1, 100.5, 98.9, 98.9],
    '두께_nm': [100.4, 102.3, 98.7, 101.1, 104.8, 104.8],
    '합격여부': [1, 1, -1, 1, -1, -1]
}

process_df = pd.DataFrame(process_data)
process_df


### 실습 1 · 데이터의 전체 모습 점검하기

**목표:** 행·열 크기, 열 이름, 자료형, 결측값 개수를 한 번에 점검합니다.

아래 코드 셀의 주석을 보고 먼저 직접 작성해 보세요. 막히면 힌트를 확인하고,
마지막에 정답 예제를 실행해 결과를 비교합니다.


In [ ]:
# TODO 1: process_df의 행 수와 열 수를 출력하세요.
# TODO 2: 열 이름과 각 열의 자료형을 출력하세요.
# TODO 3: 열별 결측값 개수를 출력하세요.


<details>
<summary><strong>힌트 보기</strong></summary>

`shape`, `columns`, `dtypes`, `isna().sum()`을 차례로 사용합니다.

</details>

**정답 예제:** 먼저 직접 시도한 뒤 아래 셀을 실행하세요.


In [ ]:
print(f'행 수: {process_df.shape[0]}, 열 수: {process_df.shape[1]}')
print('\n열 이름:', process_df.columns.tolist())
print('\n자료형:')
print(process_df.dtypes)
print('\n열별 결측값:')
print(process_df.isna().sum())


### 실습 2 · 필요한 데이터만 선택하기

**목표:** 불합격 데이터와 고온 공정 데이터를 조건식으로 골라냅니다.

아래 코드 셀의 주석을 보고 먼저 직접 작성해 보세요. 막히면 힌트를 확인하고,
마지막에 정답 예제를 실행해 결과를 비교합니다.


In [ ]:
# TODO 1: 합격여부가 -1인 행에서 lot_id, 공정, 온도_섭씨만 선택하세요.
# TODO 2: 온도가 300도 이상인 행을 선택하세요.
# TODO 3: 공정이 '증착'이면서 두께가 103nm 이상인 행을 선택하세요.


<details>
<summary><strong>힌트 보기</strong></summary>

조건마다 괄호를 쓰고, 두 조건을 모두 만족시킬 때는 `&`를 사용합니다.

</details>

**정답 예제:** 먼저 직접 시도한 뒤 아래 셀을 실행하세요.


In [ ]:
failed_lots = process_df.loc[
    process_df['합격여부'] == -1,
    ['lot_id', '공정', '온도_섭씨']
]
high_temperature = process_df[process_df['온도_섭씨'] >= 300]
deposition_risk = process_df[
    (process_df['공정'] == '증착') & (process_df['두께_nm'] >= 103)
]

print('불합격 Lot:')
print(failed_lots)
print('\n300도 이상 공정:', len(high_temperature), '건')
print('고두께 증착 공정:', len(deposition_risk), '건')


### 실습 3 · 결측값과 중복 데이터 정리하기

**목표:** 원본을 보존하면서 결측값을 중앙값으로 채우고 중복 행을 제거합니다.

아래 코드 셀의 주석을 보고 먼저 직접 작성해 보세요. 막히면 힌트를 확인하고,
마지막에 정답 예제를 실행해 결과를 비교합니다.


In [ ]:
# TODO 1: process_df를 practice_clean으로 복사하세요.
# TODO 2: 온도_섭씨의 결측값을 해당 열의 중앙값으로 채우세요.
# TODO 3: 중복 행을 제거하고 인덱스를 다시 0부터 매기세요.
# TODO 4: 정리 전후 행 수와 남은 결측값 수를 출력하세요.


<details>
<summary><strong>힌트 보기</strong></summary>

`copy()`, `median()`, `fillna()`, `drop_duplicates()`, `reset_index()`를 사용합니다.

</details>

**정답 예제:** 먼저 직접 시도한 뒤 아래 셀을 실행하세요.


In [ ]:
practice_clean = process_df.copy()
temperature_median = practice_clean['온도_섭씨'].median()
practice_clean['온도_섭씨'] = practice_clean['온도_섭씨'].fillna(temperature_median)
before_rows = len(practice_clean)
practice_clean = practice_clean.drop_duplicates().reset_index(drop=True)

print(f'정리 전 {before_rows}행 -> 정리 후 {len(practice_clean)}행')
print(f'남은 결측값: {practice_clean.isna().sum().sum()}개')
practice_clean


### 실습 4 · 새 열을 만들고 위험 순서로 정렬하기

**목표:** 기준값에서 얼마나 벗어났는지 계산하고 확인 우선순위를 정합니다.

아래 코드 셀의 주석을 보고 먼저 직접 작성해 보세요. 막히면 힌트를 확인하고,
마지막에 정답 예제를 실행해 결과를 비교합니다.


In [ ]:
# TODO 1: 두께_nm과 기준값 100의 차이를 절댓값으로 계산해 두께편차_nm 열을 만드세요.
# TODO 2: 합격여부를 {1: '합격', -1: '불합격'}으로 바꾼 판정 열을 만드세요.
# TODO 3: 두께편차_nm이 큰 순서로 정렬해 상위 3행을 확인하세요.


<details>
<summary><strong>힌트 보기</strong></summary>

절댓값은 `.abs()`, 값 치환은 `.map()`, 정렬은 `sort_values()`를 사용합니다.

</details>

**정답 예제:** 먼저 직접 시도한 뒤 아래 셀을 실행하세요.


In [ ]:
practice_clean['두께편차_nm'] = (practice_clean['두께_nm'] - 100).abs()
practice_clean['판정'] = practice_clean['합격여부'].map({1: '합격', -1: '불합격'})

priority_lots = practice_clean.sort_values(
    '두께편차_nm', ascending=False
).head(3)
priority_lots[['lot_id', '공정', '두께_nm', '두께편차_nm', '판정']]


### 실습 5 · 공정별 요약표와 그래프 만들기

**목표:** 공정별 평균 온도와 평균 두께를 계산하고 막대그래프로 비교합니다.

아래 코드 셀의 주석을 보고 먼저 직접 작성해 보세요. 막히면 힌트를 확인하고,
마지막에 정답 예제를 실행해 결과를 비교합니다.


In [ ]:
# TODO 1: 공정별 평균 온도와 평균 두께를 계산해 process_summary에 저장하세요.
# TODO 2: 평균 두께를 막대그래프로 그리세요.
# TODO 3: 제목, x축 이름, y축 이름을 추가하세요.


<details>
<summary><strong>힌트 보기</strong></summary>

`groupby()[[열 목록]].mean()`으로 요약한 뒤 `plot(kind='bar')`를 사용할 수 있습니다.

</details>

**정답 예제:** 먼저 직접 시도한 뒤 아래 셀을 실행하세요.


In [ ]:
process_summary = (
    practice_clean.groupby('공정')[['온도_섭씨', '두께_nm']]
    .mean()
    .round(2)
)
print(process_summary)

In [ ]:
process_summary['두께_nm'].plot(kind='bar', color='steelblue', figsize=(7, 4))
plt.title('공정별 평균 두께')
plt.xlabel('공정')
plt.ylabel('평균 두께 (nm)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 마무리

- 데이터 점검 → 선택 → 정리 → 새 열·정렬 → 요약·그래프 순서로 공정 데이터를 분석했습니다.
- 원본은 `copy()`로 보존하고, 결과는 행 수와 결측값 수로 검증했습니다.

다음 노트북(`05_AI_미니프로젝트.ipynb`)에서 배운 내용을 AI와 함께 복습하고 응용해 봅니다.